In [9]:
import cv2
from pathlib import Path
from datetime import timedelta
from fastapi import UploadFile
from typing import BinaryIO
from fractions import Fraction
import ffmpeg 
import hashlib

In [13]:
import subprocess
def get_video_duration_cv2(path: str) -> str:
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video file: {path}")
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
    cap.release()

    if fps <= 0:
        raise ValueError("Invalid FPS retrieved; possibly corrupted file.")

    duration_seconds = frame_count / fps
    td = timedelta(seconds=duration_seconds)

    total_seconds = td.total_seconds()
    hours, remainder = divmod(int(total_seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    milliseconds = int((total_seconds - int(total_seconds)) * 1000)
    
    return f"{hours:02}:{minutes:02}:{seconds:02}.{milliseconds:03d}"

def get_video_fps(video_path: str) -> float:
    try:
        result = subprocess.run(
            [
                "ffprobe",
                "-v", "error",
                "-select_streams", "v:0",
                "-show_entries", "stream=r_frame_rate",
                "-of", "default=noprint_wrappers=1:nokey=1",
                video_path,
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True,
        )

        rate = result.stdout.strip()  
        if "/" in rate:
            num, den = rate.split("/")
            fps = float(num) / float(den)
        else:
            fps = float(rate)

        return fps

    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Failed to get FPS via ffprobe: {e.stderr.strip()}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error while getting FPS: {e}") from e



def get_video_duration_ffprobe(path: str) -> str:
    try:
        result = subprocess.run(
            [
                "ffprobe",
                "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                path,
            ],
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            check=True,
        )

        duration_str = result.stdout.strip()
        if not duration_str:
            raise ValueError("ffprobe returned no duration; possibly corrupted file.")
        duration_seconds = float(duration_str)

        td = timedelta(seconds=duration_seconds)
        total_seconds = td.total_seconds()
        hours, remainder = divmod(int(total_seconds), 3600)
        minutes, seconds = divmod(remainder, 60)
        milliseconds = int((total_seconds - int(total_seconds)) * 1000)
        return f"{hours:02}:{minutes:02}:{seconds:02}.{milliseconds:03d}"
    except subprocess.CalledProcessError as e:
        raise RuntimeError(f"Failed to get duration via ffprobe: {e.stderr.strip()}") from e
    except Exception as e:
        raise RuntimeError(f"Unexpected error while getting duration: {e}") from e


def get_video_metadata(video_filename: str, file: BinaryIO) -> dict:
    extension = Path(video_filename).suffix
    file.seek(0)
    import tempfile
    with tempfile.NamedTemporaryFile(delete=True, suffix=extension) as tmp:
        tmp.write(file.read())
        tmp.flush()

        probe = ffmpeg.probe(tmp.name)
        video_stream = next(
            (stream for stream in probe["streams"] if stream["codec_type"] == "video"),
            None
        )
        if video_stream is None:
            return {}
        
        tmp.seek(0)
        md5 = hashlib.md5(tmp.read()).hexdigest()
        fps = float(Fraction(video_stream["avg_frame_rate"])) if "avg_frame_rate" in video_stream else None

        file.seek(0)
        return {
            "filename": video_filename,
            "codec": video_stream.get("codec_name"),
            "fps": fps,
            "width": video_stream.get("width"),
            "height": video_stream.get("height"),
            "size_bytes": int(probe["format"]["size"]),
            "checksum_md5": md5,
            'extension': extension
        }
    

In [14]:
fps = get_video_duration_ffprobe('../local/6_.mp4')
fps

'00:10:33.700'

In [6]:
import asyncio
import numpy as np
import subprocess
from typing import List, Optional, Tuple
def _read_frame_sequential(video_path: str, frame_index: int) -> tuple[bool, np.ndarray | None]:
    """Read frames sequentially up to frame_index; returns (success, frame)."""
    cap = cv2.VideoCapture(video_path)
    try:
        if not cap.isOpened():
            return False, None
        for _ in range(frame_index + 1):
            ok, frame = cap.read()
            if not ok:
                return False, None
        return True, frame #type:ignore
    finally:
        cap.release()

def read_frame_ffmpeg(video_path: str, frame_index: int) -> bytes:
    cmd = [
        "ffmpeg",
        "-hwaccel", "none",
        "-c:v", "libdav1d",  # or "libdav1d" for CPU
        "-i", video_path,
        "-vf", f"select=eq(n\\,{frame_index})",
        "-vframes", "1",
        "-f", "image2pipe",
        "-vcodec", "webp",
        "-"
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0:
        raise RuntimeError(f"ffmpeg failed: {result.stderr.decode()}")
    return result.stdout



async def read_frame(video_path: str, frame_index: int) -> bytes:
    """Async wrapper for frame extraction by frame index."""
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(None, read_frame_ffmpeg, video_path, frame_index)


def get_segment_frame_indices(start: int, end: int, n: int) -> list[int]:
    """Return n evenly spaced frame indices between start and end."""
    if n <= 0 or end <= start:
        return []
    total = end - start
    return [start + (i + 1) * total // (n + 1) for i in range(n)]


async def extract_frames_from_segments(
    video_path: str,
    segments: List[Tuple[int, int]],
    n_per_segment: int
) -> List[Tuple[int, bytes]]:
    all_indices = []
    for start, end in segments:
        all_indices.extend(
            get_segment_frame_indices(start,end, n_per_segment)
        )
    all_indices = sorted(set(all_indices))
    tasks = [read_frame(video_path, idx) for idx in all_indices]
    frames = await asyncio.gather(*tasks)

    return list(zip(all_indices, frames))


In [ ]:
video = await extract_frames_from_segments('../local/6_.mp4', [(20,100)], 50)